In [1]:
# Find the project root from either the workspace or notebooks folder.
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "src/task_b").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the NLP_PROJECT_task_b folder in VS Code before running this notebook")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Three-class FEVER evaluation: SUPPORTED / CONTRADICTED (gold evidence) and UNSUPPORTED (NOT ENOUGH INFO claims
# with retrieved evidence, unreviewed). Thresholds are chosen on validation; the test set is scored once.
import random
import pandas as pd
from task_b.fever import claim_key, load_source_prior, prepare_fever, retrieve_nei_evidence, write_prepared, write_validation_sample
from task_b.schema import LABELS, parse_claim_evidence
from task_b.verification import (choose_threshold, evaluate, lexical_cues, lexical_overlap, majority_prior, overlap_score,
                                 save_run, timed_predictions)
TRAIN_CLAIMS = PROJECT_ROOT / "data/raw/fever/train.jsonl"
EVAL_CLAIMS = PROJECT_ROOT / "data/raw/fever/shared_task_dev.jsonl"
WIKI_DIR = PROJECT_ROOT / "data/raw/fever/wiki-pages"
SEED, PER_LABEL, SENTENCES, MAX_PAGES, REVIEW_SAMPLE = 42, 2000, 1, 3, 100
PREPARED = PROJECT_ROOT / "outputs/task_b/fever_prepared"
RUN_DIR = PROJECT_ROOT / "outputs/task_b/fever_three_class" / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
assert (PREPARED / "test/records.jsonl").is_file(), "Run task_b_fever_baselines.ipynb first (it prepares the gold test records)"

In [2]:
# Test SUPPORTS/REFUTES claims: exactly the gold-evidence records used by the two-class runs.
gold_rows = [json.loads(line) for line in (PREPARED / "test/records.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
test_examples = [parse_claim_evidence(row["claim"]) for row in gold_rows]
test_gold = [row["gold_label"] for row in gold_rows]
test_source = ["gold"] * len(gold_rows)
print(f"Gold-evidence test claims: {len(gold_rows):,}")

Gold-evidence test claims: 13,202


In [3]:
# Test NOT ENOUGH INFO claims: evidence from title-matched Wikipedia pages (never empty). Unmatched claims are
# excluded and counted. Two Wikipedia scans; takes a few minutes.
nei_test, nei_test_report = retrieve_nei_evidence(EVAL_CLAIMS, WIKI_DIR, SENTENCES, MAX_PAGES)
write_prepared(nei_test, nei_test_report, PREPARED / f"test_nei_retrieved_s{SENTENCES}_p{MAX_PAGES}")
test_examples += [record.claim for record in nei_test]
test_gold += [record.gold_label for record in nei_test]
test_source += ["retrieved"] * len(nei_test)
print({key: nei_test_report[key] for key in ("nei_claims", "retrieved", "no_matching_page", "no_sentence_on_matched_pages")})

{'nei_claims': 6666, 'retrieved': 6552, 'no_matching_page': 114, 'no_sentence_on_matched_pages': 0}


In [4]:
# Validation: a seeded sample of training claims, 2,000 per FEVER label (none shares text with a test claim),
# prepared the same way: gold evidence for SUPPORTS/REFUTES, retrieved evidence for NOT ENOUGH INFO.
test_claim_keys = {claim_key(json.loads(line)["claim"]) for line in EVAL_CLAIMS.read_text(encoding="utf-8").splitlines() if line.strip()}
validation_dir = PREPARED / f"validation3_seed{SEED}_n{PER_LABEL}_s{SENTENCES}_p{MAX_PAGES}"
manifest = write_validation_sample(TRAIN_CLAIMS, validation_dir / "claims.jsonl", PER_LABEL, SEED, test_claim_keys, include_nei=True)
gold_validation, gold_validation_report = prepare_fever(validation_dir / "claims.jsonl", WIKI_DIR)
nei_validation, nei_validation_report = retrieve_nei_evidence(validation_dir / "claims.jsonl", WIKI_DIR, SENTENCES, MAX_PAGES)
write_prepared(gold_validation + nei_validation, {"gold": gold_validation_report, "nei": nei_validation_report, "sample": manifest}, validation_dir)
validation_examples = [record.claim for record in gold_validation + nei_validation]
validation_gold = [record.gold_label for record in gold_validation + nei_validation]
print("validation:", {"sampled": manifest["rows"], "gold resolved": gold_validation_report["resolved"],
                      "NEI retrieved": nei_validation_report["retrieved"], "NEI without page": nei_validation_report["no_matching_page"]})

validation: {'sampled': 6000, 'gold resolved': 3965, 'NEI retrieved': 1980, 'NEI without page': 20}


In [5]:
# Shortcut check: if evidence size or word overlap differed sharply by label, a model could guess the label
# without verifying anything. Compare these per label before trusting any score.
def profile(examples, gold):
    frame = pd.DataFrame({"label": gold, "sentences": [len(e.evidence) for e in examples],
                          "evidence words": [sum(len(s.text.split()) for s in e.evidence) for e in examples],
                          "claim-word overlap": [overlap_score(e) for e in examples]})
    return frame.groupby("label").agg(claims=("label", "size"), mean_sentences=("sentences", "mean"),
                                      median_evidence_words=("evidence words", "median"),
                                      median_overlap=("claim-word overlap", "median")).round(3)

pd.concat({"validation": profile(validation_examples, validation_gold), "test": profile(test_examples, test_gold)})

claims  mean_sentences  median_evidence_words  \
           label                                                         
validation CONTRADICTED    1987           1.141                   30.0   
           SUPPORTED       1978           1.199                   32.0   
           UNSUPPORTED     1980           1.000                   26.5   
test       CONTRADICTED    6605           1.118                   28.0   
           SUPPORTED       6597           1.140                   28.0   
           UNSUPPORTED     6552           1.000                   26.0   

                         median_overlap  
           label                         
validation CONTRADICTED           0.571  
           SUPPORTED              0.667  
           UNSUPPORTED            0.500  
test       CONTRADICTED           0.571  
           SUPPORTED              0.667  
           UNSUPPORTED            0.542

In [6]:
# Lexical baselines: the overlap cutoff of each is chosen by three-class validation macro-F1 (test not used).
# lexical_overlap can only say SUPPORTED or UNSUPPORTED; lexical_cues adds negation/number contradictions.
GRID = [round(0.1 * step, 1) for step in range(1, 10)]
prior = majority_prior(load_source_prior(TRAIN_CLAIMS, include_nei=True))
thresholds, searches = {}, {}
for name, function in [("lexical", lexical_overlap), ("lexical_cues", lexical_cues)]:
    thresholds[name], searches[name] = choose_threshold(
        validation_gold, lambda cutoff, f=function: [f(example, cutoff) for example in validation_examples], GRID, labels=LABELS)
    print(f"{name:<13} chosen cutoff {thresholds[name]} | validation macro-F1:",
          " ".join(f"{row['threshold']}={row['validation_macro_f1']:.3f}" for row in searches[name]))
print("Training prior:", prior)

lexical       chosen cutoff 0.6 | validation macro-F1: 0.1=0.168 0.2=0.187 0.3=0.238 0.4=0.280 0.5=0.311 0.6=0.330 0.7=0.325 0.8=0.304 0.9=0.244


lexical_cues  chosen cutoff 0.5 | validation macro-F1: 0.1=0.297 0.2=0.313 0.3=0.356 0.4=0.395 0.5=0.413 0.6=0.411 0.7=0.382 0.8=0.348 0.9=0.266
Training prior: SUPPORTED


In [7]:
# Embedding baseline: MiniLM cosine similarity >= cutoff -> SUPPORTED, otherwise UNSUPPORTED (it cannot output
# CONTRADICTED). The cutoff is chosen by three-class validation macro-F1. Uses the cached model; never downloads.
import warnings
import transformers
warnings.filterwarnings("ignore", message="IProgress not found")
transformers.logging.set_verbosity_error()
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
from task_b.models import embedding_similarities, model_fingerprint
EMBEDDING_DIR = PROJECT_ROOT / "models/task_b/all-MiniLM-L6-v2"
if not EMBEDDING_DIR.is_dir():
    EMBEDDING_DIR = Path(snapshot_download("sentence-transformers/all-MiniLM-L6-v2", local_files_only=True))
encoder = SentenceTransformer(str(EMBEDDING_DIR))
similarity = {"validation": embedding_similarities(encoder, validation_examples), "test": embedding_similarities(encoder, test_examples)}
def labels_at(scores, cutoff):
    return ["SUPPORTED" if score >= cutoff else "UNSUPPORTED" for score in scores]
thresholds["embedding"], searches["embedding"] = choose_threshold(
    validation_gold, lambda cutoff: labels_at(similarity["validation"], cutoff), [round(0.05 * s, 2) for s in range(4, 20)], labels=LABELS)
print("embedding chosen cutoff", thresholds["embedding"])

C:\Users\ONYX\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7303.81it/s]

embedding chosen cutoff 0.65


In [8]:
# Score the test set once with every method; save each run separately with its coverage.
methods = {"majority": lambda: [prior] * len(test_examples),
           "lexical": lambda: [lexical_overlap(e, thresholds["lexical"]) for e in test_examples],
           "lexical_cues": lambda: [lexical_cues(e, thresholds["lexical_cues"]) for e in test_examples],
           "embedding": lambda: labels_at(similarity["test"], thresholds["embedding"])}
coverage = {"test gold-evidence claims": len(gold_rows), "test NEI claims with retrieved evidence": len(nei_test),
            "test NEI claims excluded (no page)": nei_test_report["no_matching_page"] + nei_test_report["no_sentence_on_matched_pages"],
            "validation claims": len(validation_examples)}
results = {}
for name, predict in methods.items():
    predictions = predict()
    assert len(predictions) == len(test_gold), f"{name} did not score every test claim"
    score = evaluate(test_gold, predictions, LABELS)
    score["evaluation_scope"] = "FEVER three-class; NEI evidence retrieved and unreviewed"
    results[name] = score
    save_run(RUN_DIR / name, {"method": name, "threshold": thresholds.get(name), "validation_search": searches.get(name),
                              "coverage": coverage, "nei_policy": nei_test_report["policy"], "sentences_per_claim": SENTENCES,
                              "max_pages": MAX_PAGES, "validation_manifest": manifest,
                              "embedding_model_sha256": model_fingerprint(EMBEDDING_DIR) if name == "embedding" else None},
             [{"claim_id": e.claim_id, "gold": g, "evidence_source": s, "prediction": p}
              for e, g, s, p in zip(test_examples, test_gold, test_source, predictions)], score)
print(coverage)

{'test gold-evidence claims': 13202, 'test NEI claims with retrieved evidence': 6552, 'test NEI claims excluded (no page)': 114, 'validation claims': 5945}


In [9]:
# NLI (evidence = premise, claim = hypothesis) gives all three labels; it runs only if a local RoBERTa-MNLI snapshot exists.
NLI_DIR = PROJECT_ROOT / "models/task_b/roberta-large-mnli"
if NLI_DIR.is_dir():
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from task_b.models import nli_predictor
    tokenizer = AutoTokenizer.from_pretrained(str(NLI_DIR), local_files_only=True)
    nli_model = AutoModelForSequenceClassification.from_pretrained(str(NLI_DIR), local_files_only=True)
    predictions, latency = timed_predictions(test_examples, nli_predictor(tokenizer, nli_model, torch))
    results["nli"] = evaluate(test_gold, predictions, LABELS, latency)
    save_run(RUN_DIR / "nli", {"method": "nli", "model_path": str(NLI_DIR), "model_snapshot_sha256": model_fingerprint(NLI_DIR),
                               "coverage": coverage, "aggregation": "project-specific per-sentence rule"},
             [{"claim_id": e.claim_id, "gold": g, "evidence_source": s, "prediction": p}
              for e, g, s, p in zip(test_examples, test_gold, test_source, predictions)], results["nli"])
else:
    print(f"NLI skipped: no local model at {NLI_DIR.relative_to(PROJECT_ROOT)} (download not approved yet).")

NLI skipped: no local model at models\task_b\roberta-large-mnli (download not approved yet).


In [10]:
# Three-class test results: overall, per class, and confusion matrices (rows = gold, columns = predicted).
display(pd.DataFrame([{"method": name, "n": s["n"], "accuracy": s["accuracy"], "macro_f1": s["macro_f1"],
                       "wrong_SUPPORTED": s["incorrect_supported"], "CONTRADICTED_recall": s["contradicted_recall"]}
                      for name, s in results.items()]).round(3))
for name, s in results.items():
    print(name)
    display(pd.DataFrame(s["per_class"]).T.round(3))
    display(pd.DataFrame(s["confusion_matrix"], index=[f"gold {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS]))

,method,n,accuracy,macro_f1,wrong_SUPPORTED,CONTRADICTED_recall
0,majority,19754,0.334,0.167,13157,0.000
1,lexical,19754,0.405,0.324,6000,0.000
2,lexical_cues,19754,0.429,0.390,7419,0.149
3,embedding,19754,0.411,0.329,4346,0.000


majority


,precision,recall,f1,support
SUPPORTED,0.334,1.0,0.501,6597.0
CONTRADICTED,0.000,0.0,0.000,6605.0
UNSUPPORTED,0.000,0.0,0.000,6552.0


,pred SUPPORTED,pred CONTRADICTED,pred UNSUPPORTED
gold SUPPORTED,6597,0,0
gold CONTRADICTED,6605,0,0
gold UNSUPPORTED,6552,0,0


lexical


...

,precision,recall,f1,support
SUPPORTED,0.402,0.755,0.524,6597.0
CONTRADICTED,0.694,0.149,0.245,6605.0
UNSUPPORTED,0.422,0.383,0.401,6552.0


,pred SUPPORTED,pred CONTRADICTED,pred UNSUPPORTED
gold SUPPORTED,4979,177,1441
gold CONTRADICTED,3630,984,1991
gold UNSUPPORTED,3789,256,2507


embedding


,precision,recall,f1,support
SUPPORTED,0.455,0.550,0.498,6597.0
CONTRADICTED,0.000,0.000,0.000,6605.0
UNSUPPORTED,0.381,0.685,0.490,6552.0


,pred SUPPORTED,pred CONTRADICTED,pred UNSUPPORTED
gold SUPPORTED,3631,0,2966
gold CONTRADICTED,2285,0,4320
gold UNSUPPORTED,2061,0,4491


In [11]:
# Human-review sheet for the retrieved NEI evidence: a seeded sample with blank label columns. Reviewers decide whether
# the evidence really leaves each claim unsupported; nothing is filled in here.
review = random.Random(SEED).sample(nei_test, min(REVIEW_SAMPLE, len(nei_test)))
pd.DataFrame([{"claim_id": r.claim.claim_id, "claim": r.claim.claim, "evidence_page": r.claim.evidence[0].source_document_id,
               "evidence_text": r.claim.evidence[0].text, "reviewer_id": "", "reviewer_label": "", "rationale": ""}
              for r in review]).to_csv(RUN_DIR / "nei_evidence_review_sheet.csv", index=False)
print(f"Review sheet with {len(review)} blank rows saved to", (RUN_DIR / "nei_evidence_review_sheet.csv").relative_to(PROJECT_ROOT))

Review sheet with 100 blank rows saved to outputs\task_b\fever_three_class\run_20261022T010325Z\nei_evidence_review_sheet.csv


In [12]:
# Summary from the measured scores, and the limits of this evaluation.
best = max(results, key=lambda name: results[name]["macro_f1"])
print(f"Highest three-class test macro-F1: {best} ({results[best]['macro_f1']:.3f}) on {results[best]['n']:,} claims "
      f"({len(gold_rows):,} gold-evidence SUPPORTS/REFUTES + {len(nei_test):,} NEI with retrieved evidence).")
print("Limits: NEI evidence is retrieved automatically and not yet human-reviewed; SUPPORTS/REFUTES use gold evidence while NEI "
      "uses retrieved evidence, so check the shortcut table above; keep these results separate from the two-class diagnostic.")

Highest three-class test macro-F1: lexical_cues (0.390) on 19,754 claims (13,202 gold-evidence SUPPORTS/REFUTES + 6,552 NEI with retrieved evidence).
Limits: NEI evidence is retrieved automatically and not yet human-reviewed; SUPPORTS/REFUTES use gold evidence while NEI uses retrieved evidence, so check the shortcut table above; keep these results separate from the two-class diagnostic.
